In [30]:
!pip install -q catboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [32]:
# 한글 폰트 설정
!pip install koreanize-matplotlib -q

try:
    import koreanize_matplotlib
except ImportError:
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
#시드 고정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [34]:
TARGET = "임신 성공 여부"
ID_COL = "ID"
DATA_DIR = Path(".")   # 데이터 경로 수정

SEED = 42

In [40]:
AGE_MAP = {
    "만18-34세": 0, "만35-37세": 1, "만38-39세": 2,
    "만40-42세": 3, "만43-44세": 4, "만45-50세": 5,
    "알 수 없음": -1,
}
COUNT_MAP = {"0회": 0, "1회": 1, "2회": 2, "3회": 3, "4회": 4, "5회": 5, "6회 이상": 6}
DONOR_AGE_MAP = {
    "만20세 이하": 0, "만21-25세": 1, "만26-30세": 2,
    "만31-35세": 3,  "만36-40세": 4, "만41-45세": 5,
    "알 수 없음": -1,
}
COUNT_COLS = [
    "총 시술 횟수", "클리닉 내 총 시술 횟수",
    "IVF 시술 횟수", "DI 시술 횟수",
    "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수",
]

In [41]:
DROP_COLS = [
    "착상 전 유전 검사 사용 여부",   # 98.94% 결측, 비결측값 모두 1.0
    "PGD 시술 여부",                 # 99.15% 결측
    "PGS 시술 여부",                 # 99.25% 결측
    "불임 원인 - 여성 요인",          # zero variance
    "난자 채취 경과일",               # zero variance
    "난자 해동 경과일",               # 99.44% 결측
]

In [42]:
REASON_CATEGORIES = ["현재 시술용", "배아 저장용", "난자 저장용", "기증용", "연구용"]
PROCEDURE_TYPES   = ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER",
                     "BLASTOCYST", "AH", "Generic DI", "IVI"]

def expand_reason(df):
    col = "배아 생성 주요 이유"
    for cat in REASON_CATEGORIES:
        df[f"이유_{cat}"] = df[col].fillna("").str.contains(cat).astype(int)
    return df

def expand_procedure(df):
    col = "특정 시술 유형"
    filled = df[col].fillna("Unknown")
    for pt in PROCEDURE_TYPES:
        df[f"시술_{pt}"] = (
            filled.str.upper().str.replace(" ", "", regex=False)
            .str.contains(pt.upper()).astype(int)
        )
    return df

In [43]:
def preprocess(df: pd.DataFrame, fit_label_encoders=None) -> tuple:
    df = df.copy()

    # 컬럼 제거
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

    # 임신 시도 연수 (96% 결측) → 결측 지시 변수 + -1 대체
    col_years = "임신 시도 또는 마지막 임신 경과 연수"
    if col_years in df.columns:
        df["임신_시도_연수_결측"] = df[col_years].isnull().astype(int)
        df[col_years] = df[col_years].fillna(-1)

    # 배아 해동 경과일 (84% 결측) → 결측 지시 변수 + 0 대체
    col_thaw = "배아 해동 경과일"
    if col_thaw in df.columns:
        df["배아_해동_결측"] = df[col_thaw].isnull().astype(int)
        df[col_thaw] = df[col_thaw].fillna(0)

    # DI 시술행 IVF 전용 컬럼 → 0 대체
    FILL_ZERO_COLS = [
        "단일 배아 이식 여부", "착상 전 유전 진단 사용 여부", "배아 생성 주요 이유",
        "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수",
        "이식된 배아 수", "미세주입 배아 이식 수", "저장된 배아 수",
        "미세주입 후 저장된 배아 수", "해동된 배아 수", "해동 난자 수",
        "수집된 신선 난자 수", "저장된 신선 난자 수", "혼합된 난자 수",
        "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",
        "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "대리모 여부",
    ]
    for col in FILL_ZERO_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 경과일 컬럼 중앙값 대체
    for col in ["난자 혼합 경과일", "배아 이식 경과일"]:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # 순서형 인코딩: 나이
    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(AGE_MAP).fillna(-1).astype(int)

    # 순서형 인코딩: 횟수 컬럼
    for col in COUNT_COLS:
        if col in df.columns:
            df[col] = df[col].map(COUNT_MAP).fillna(-1).astype(int)

    # 순서형 인코딩: 기증자 나이
    for col in ["난자 기증자 나이", "정자 기증자 나이"]:
        if col in df.columns:
            df[col] = df[col].map(DONOR_AGE_MAP).fillna(-1).astype(int)

    # 시술 유형 이진화 (IVF=1, DI=0)
    if "시술 유형" in df.columns:
        df["시술 유형"] = (df["시술 유형"] == "IVF").astype(int)

    # 배란 유도 유형 인코딩
    induction_map = {
        "알 수 없음": 0, "기록되지 않은 시행": 1,
        "생식선 자극 호르몬": 2, "세트로타이드 (억제제)": 3,
    }
    if "배란 유도 유형" in df.columns:
        df["배란 유도 유형"] = df["배란 유도 유형"].map(induction_map).fillna(0).astype(int)

    # 난자/정자 출처 인코딩
    egg_map   = {"본인 제공": 0, "기증 제공": 1, "알 수 없음": -1}
    sperm_map = {"배우자 제공": 0, "기증 제공": 1, "배우자 및 기증 제공": 2, "미할당": -1}
    if "난자 출처" in df.columns:
        df["난자 출처"] = df["난자 출처"].map(egg_map).fillna(-1).astype(int)
    if "정자 출처" in df.columns:
        df["정자 출처"] = df["정자 출처"].map(sperm_map).fillna(-1).astype(int)

    # 시술 시기 코드 레이블 인코딩
    code_map = {v: i for i, v in enumerate(
        ["TRCMWS", "TRDQAZ", "TRJXFG", "TRVNRY", "TRXQMD", "TRYBLT", "TRZKPL"]
    )}
    if "시술 시기 코드" in df.columns:
        df["시술 시기 코드"] = df["시술 시기 코드"].map(code_map).fillna(-1).astype(int)

    # 레이블 인코딩: 명목형 범주  ← project_1.ipynb
    LABEL_ENC_COLS = ["배아 생성 주요 이유", "특정 시술 유형"]
    label_encoders = fit_label_encoders or {}
    for col in LABEL_ENC_COLS:
        if col not in df.columns:
            continue
        if col not in label_encoders:
            le = LabelEncoder()
            le.fit(df[col].fillna("Unknown").astype(str))
            label_encoders[col] = le
        df[col] = label_encoders[col].transform(df[col].fillna("Unknown").astype(str))

    # 이진 컬럼 결측 → 0
    binary_cols = [
        "배란 자극 여부", "단일 배아 이식 여부", "착상 전 유전 진단 사용 여부",
        "남성 주 불임 원인", "남성 부 불임 원인",
        "여성 주 불임 원인", "여성 부 불임 원인",
        "부부 주 불임 원인", "부부 부 불임 원인", "불명확 불임 원인",
        "불임 원인 - 난관 질환", "불임 원인 - 남성 요인", "불임 원인 - 배란 장애",
        "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증",
        "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인",
        "불임 원인 - 정자 운동성", "불임 원인 - 정자 형태",
        "동결 배아 사용 여부", "신선 배아 사용 여부",
        "기증 배아 사용 여부", "대리모 여부",
    ]
    for c in binary_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    # 연속형 수치 결측 → 중앙값
    cont_cols = [
        "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수",
        "이식된 배아 수", "미세주입 배아 이식 수",
        "저장된 배아 수", "미세주입 후 저장된 배아 수",
        "해동된 배아 수", "해동 난자 수",
        "수집된 신선 난자 수", "저장된 신선 난자 수",
        "혼합된 난자 수", "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",
        "난자 혼합 경과일", "배아 이식 경과일", "배아 해동 경과일",
    ]
    for c in cont_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df[c] = df[c].fillna(df[c].median() if df[c].notna().any() else -1)

    return df, label_encoders


# 멀티레이블 분리는 전처리 전에 적용
def preprocess_full(df, fit_label_encoders=None):
    df = expand_reason(df.copy())
    df = expand_procedure(df)
    df.drop(columns=["배아 생성 주요 이유", "특정 시술 유형"],
            inplace=True, errors="ignore")
    df, le = preprocess(df, fit_label_encoders)
    return df, le


X_raw = train.drop(columns=[ID_COL, TARGET], errors="ignore")
y     = train[TARGET]
X_test_raw = test.drop(columns=[ID_COL], errors="ignore")

X_pp, label_encoders = preprocess_full(X_raw)
X_test_pp, _         = preprocess_full(X_test_raw, fit_label_encoders=label_encoders)

print(f"\n  전처리 후 Train shape: {X_pp.shape}")
print(f"  전처리 후 Test  shape: {X_test_pp.shape}")
print(f"  잔여 결측치 (Train): {X_pp.isnull().sum().sum()}")


  전처리 후 Train shape: (256351, 76)
  전처리 후 Test  shape: (90067, 76)
  잔여 결측치 (Train): 0


피처 엔지니어링

In [48]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 불임 원인 개수 합산
    cause_cols = [c for c in df.columns if "불임 원인 -" in c]
    df["불임_원인_합계"] = df[cause_cols].sum(axis=1)

    # 주요 불임 원인 합계
    primary_cols = [c for c in ["남성 주 불임 원인", "여성 주 불임 원인", "부부 주 불임 원인"]
                    if c in df.columns]
    df["주요_불임_원인_합계"] = df[primary_cols].sum(axis=1)

    # 이식 효율: 이식 배아 / 생성 배아
    df["이식_효율"] = np.where(
        df["총 생성 배아 수"] > 0,
        df["이식된 배아 수"] / df["총 생성 배아 수"], 0
    )

    # ICSI 비율
    if "미세주입에서 생성된 배아 수" in df.columns:
        df["ICSI_비율"] = np.where(
            df["총 생성 배아 수"] > 0,
            df["미세주입에서 생성된 배아 수"] / df["총 생성 배아 수"], 0
        )

    # 과거 임신 성공률
    if "총 임신 횟수" in df.columns and "총 시술 횟수" in df.columns:
        df["과거_임신_성공률"] = np.where(
            df["총 시술 횟수"] > 0,
            df["총 임신 횟수"] / (df["총 시술 횟수"] + 1), 0
        )

    # 배아 전략 플래그 (동결=1, 기증=2, 신선=0)
    for col in ["동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부"]:
        if col not in df.columns:
            df[col] = 0
    df["배아_전략"] = (
        df["동결 배아 사용 여부"] * 1
        + df["신선 배아 사용 여부"] * 0
        + df["기증 배아 사용 여부"] * 2
    )

    # 저장 비율
    df["저장_비율"] = np.where(
        df["총 생성 배아 수"] > 0,
        df["저장된 배아 수"] / df["총 생성 배아 수"], 0
    )

    # 기증 사용 플래그
    df["기증_사용"] = (
        (df.get("난자 출처", pd.Series(0, index=df.index)) == 1)
        | (df.get("정자 출처", pd.Series(0, index=df.index)) == 1)
    ).astype(int)

      # 동결배아 활용률: 실제 해동 사용 / 저장 총수
    #   → 높을수록 보관 배아를 적극 활용한 주기
    if "해동된 배아 수" in df.columns:
        df["동결배아_활용률"] = np.where(
            df["저장된 배아 수"] > 0,
            df["해동된 배아 수"] / df["저장된 배아 수"], 0
        )

    # ICSI 후 동결된 배아 비율: ICSI 저장 / 이식 배아
    #   → 미세주입 배아 중 실제 이식 외 저장된 비중
    if "미세주입 후 저장된 배아 수" in df.columns:
        df["ICSI동결_이식비율"] = np.where(
            df["이식된 배아 수"] > 0,
            df["미세주입 후 저장된 배아 수"] / df["이식된 배아 수"], 0
        )

    # 순수 동결 주기 플래그: 동결=1 & 신선=0
    #   → 신선 배아 없이 동결만 사용한 주기 (FER 주기)
    df["순수동결_주기"] = (
        (df["동결 배아 사용 여부"] == 1) & (df["신선 배아 사용 여부"] == 0)
    ).astype(int)

    # 동결 잉여율: (저장 - 해동) / 저장
    #   → 아직 남아있는 동결 배아 비율 (낮을수록 소진됨)
    if "해동된 배아 수" in df.columns:
        remaining = (df["저장된 배아 수"] - df["해동된 배아 수"]).clip(lower=0)
        df["동결배아_잉여율"] = np.where(
            df["저장된 배아 수"] > 0,
            remaining / df["저장된 배아 수"], 0
        )

    # 동결 배아 총량 지수: 저장 + 해동 (해동 포함 시)
    #   → 누적 동결 배아 규모를 반영
    if "해동된 배아 수" in df.columns:
        df["동결배아_총량"] = df["저장된 배아 수"] + df["해동된 배아 수"]
    # 전처리 후 "시술 당시 나이"는 순서형 정수 0~5 (-1=알 수 없음)
    # "수집된 신선 난자 수"는 연속형 수치

    if "수집된 신선 난자 수" in df.columns and "시술 당시 나이" in df.columns:
        age_num = df["시술 당시 나이"].replace(-1, np.nan)  # 알 수 없음 → NaN
        egg_cnt = df["수집된 신선 난자 수"]

        # 나이 × 난자 수 교호작용항
        #   → 나이 코드가 높고 난자도 많은 경우 큰 값 (고령 고수확)
        df["나이x난자수"] = (age_num * egg_cnt).fillna(0)

        # 나이 보정 난자 효율: 난자 수 / (나이 코드 + 1)
        #   → 같은 난자 수라도 나이가 낮을수록 효율이 높음을 표현
        df["나이보정_난자효율"] = (egg_cnt / (age_num.fillna(0) + 1))

        # 혼합 난자 대비 수정 효율: 총 배아 / 혼합 난자
        #   → 난자 수정 성공률 근사
        if "혼합된 난자 수" in df.columns:
            df["수정_효율"] = np.where(
                df["혼합된 난자 수"] > 0,
                df["총 생성 배아 수"] / df["혼합된 난자 수"], 0
            )

        # 고령 저반응 플래그: 나이 >= 3 (만40-42세+) AND 난자 수 < 중앙값
        #   → 고령임에도 채취 수가 낮은 불량 예후 케이스
        egg_median = egg_cnt.median()
        df["고령저반응"] = (
            (age_num >= 3) & (egg_cnt < egg_median)
        ).fillna(False).astype(int)

        # 젊고 고수확 플래그: 나이 <= 1 (만18-37세) AND 난자 수 >= 75th 백분위
        egg_q75 = egg_cnt.quantile(0.75)
        df["젊고고수확"] = (
            (age_num <= 1) & (egg_cnt >= egg_q75)
        ).fillna(False).astype(int)
      # 배아 손실 수: 생성 - 이식 - 저장 (폐기된 배아 추정)
    #   → 낮을수록 배아 관리 효율이 좋음
    df["배아_손실수"] = (
        df["총 생성 배아 수"]
        - df["이식된 배아 수"]
        - df["저장된 배아 수"]
    ).clip(lower=0)

    # 배아 총 활용률: (이식 + 저장) / 생성
    #   → 1에 가까울수록 생성 배아를 모두 활용
    df["배아_총활용률"] = np.where(
        df["총 생성 배아 수"] > 0,
        (df["이식된 배아 수"] + df["저장된 배아 수"]) / df["총 생성 배아 수"], 0
    ).clip(0, 1)

    # 배아 풍요도 구간: 0=없음, 1=적음(1-3), 2=보통(4-7), 3=많음(8+)
    embryo_max = max(df["총 생성 배아 수"].max() + 1, 9)
    df["배아_풍요도"] = pd.cut(
        df["총 생성 배아 수"],
        bins=[-1, 0, 3, 7, embryo_max],
        labels=[0, 1, 2, 3]
    ).astype(float).fillna(0).astype(int)

    # 다배아 이식 플래그: 이식 배아 >= 3 (다태 임신 위험 기준)
    df["다배아이식_플래그"] = (df["이식된 배아 수"] >= 3).astype(int)

    # 배아 품질 복합 지수: 이식 효율 × 배아 총 활용률
    #   → 두 효율 지표를 동시에 고려한 종합 배아 품질 점수 (0~1)
    df["배아품질_복합지수"] = df["이식_효율"] * df["배아_총활용률"]

    # ICSI 배아 우위 비율: ICSI 배아 / 총 배아
    #   → 미세주입 의존도. 높을수록 정자 문제 가능성
    if "미세주입에서 생성된 배아 수" in df.columns:
        df["ICSI배아_우위비율"] = np.where(
            df["총 생성 배아 수"] > 0,
            df["미세주입에서 생성된 배아 수"] / df["총 생성 배아 수"], 0
        )

    # 이식 집중도: 이식 배아 / (이식 + 저장)
    #   → 보관보다 즉시 이식에 집중한 정도
    denom_43 = df["이식된 배아 수"] + df["저장된 배아 수"]
    df["이식_집중도"] = np.where(denom_43 > 0, df["이식된 배아 수"] / denom_43, 0)

    if "시술 유형" in df.columns and "시술 당시 나이" in df.columns:
        age_num2 = df["시술 당시 나이"].replace(-1, 0)  # 알 수 없음 → 0

        # 나이 × 시술 유형 교호작용항
        #   → IVF 시술 + 나이가 높을수록 큰 값. IVF의 나이 의존성 포착
        df["나이x시술유형"] = age_num2 * df["시술 유형"]

        # 고령 IVF 플래그: 나이 >= 3 (만40+) AND IVF 시술
        #   → 고령에서 IVF를 선택한 고위험 케이스
        df["고령IVF"] = (
            (df["시술 당시 나이"] >= 3) & (df["시술 유형"] == 1)
        ).astype(int)

        # 젊은 DI 플래그: 나이 <= 1 (만18-37세) AND DI 시술
        #   → 상대적으로 단순 시술로도 기대치가 높은 케이스
        df["젊은DI"] = (
            (df["시술 당시 나이"] <= 1) & (df["시술 유형"] == 0)
        ).astype(int)

        # 나이 구간별 시술 유형 복합 코드: age_band * 10 + procedure_type
        #   → 나이-시술 조합을 단일 범주로 표현 (트리 모델에서 분기 포착)
        df["나이_시술_복합코드"] = df["시술 당시 나이"].clip(lower=0) * 10 + df["시술 유형"]

    if "총 시술 횟수" in df.columns and "시술 당시 나이" in df.columns:
        age_num3  = df["시술 당시 나이"].replace(-1, 0)
        trial_num = df["총 시술 횟수"].replace(-1, 0)

        # 나이 × 총 시술 횟수 교호작용항
        #   → 나이가 많고 시술 횟수가 많을수록 큰 값 (반복 실패 + 고령)
        df["나이x총시술횟수"] = age_num3 * trial_num

        # 클리닉 내 시술 횟수 교호작용 (동일 클리닉 내 축적 경험)
        if "클리닉 내 총 시술 횟수" in df.columns:
            clinic_num = df["클리닉 내 총 시술 횟수"].replace(-1, 0)
            df["나이x클리닉시술횟수"] = age_num3 * clinic_num

        # IVF 전용 시술 횟수 × 나이
        if "IVF 시술 횟수" in df.columns:
            ivf_num = df["IVF 시술 횟수"].replace(-1, 0)
            df["나이xIVF횟수"] = age_num3 * ivf_num

        # 고령 반복 시술 플래그: 나이 >= 3 AND 총 시술 횟수 >= 3
        #   → 고령 + 반복 실패 이력의 복합 고위험군
        df["고령반복시술"] = (
            (df["시술 당시 나이"] >= 3) & (trial_num >= 3)
        ).astype(int)

        # 시술 누적 부담 지수: 나이 × (총 시술 + IVF 시술) / 2
        #   → 나이와 시술 이력을 결합한 누적 부담 점수
        if "IVF 시술 횟수" in df.columns:
            ivf_num = df["IVF 시술 횟수"].replace(-1, 0)
            df["시술부담_지수"] = age_num3 * (trial_num + ivf_num) / 2

    print(f"  피처 수: {df.shape[1]}개")
    return df

X_train = feature_engineering(X_pp)
X_test  = feature_engineering(X_test_pp)

  피처 수: 110개
  피처 수: 110개


In [49]:
common_cols = [c for c in X_train.columns if c in X_test.columns]
X_train = X_train[common_cols]
X_test  = X_test[common_cols]

print(f"  최종 Train 피처: {X_train.shape[1]}개")
print(f"  최종 Test  피처: {X_test.shape[1]}개")

  최종 Train 피처: 110개
  최종 Test  피처: 110개


모델 학습

In [50]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(X_train))
oof_xgb  = np.zeros(len(X_train))
oof_cat  = np.zeros(len(X_train))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

In [51]:
# ============================================================
# 7. LightGBM 단독 모델 학습 (5-Fold Stratified CV)
# ============================================================
print("\n" + "=" * 60)
print("7. LightGBM 단독 모델 학습 (5-Fold Stratified CV)")
print("=" * 60)

N_SPLITS = 5
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_pred  = np.zeros(len(X_train))   # OOF 예측 확률
test_pred = np.zeros(len(X_test))    # 테스트 예측 확률 (폴드 평균)
fold_aucs = []                        # 폴드별 AUC 기록
lgb_models = []                       # 폴드별 학습된 lgb 모델 보관

lgb_params = {
    "objective":         "binary",
    "metric":            "auc",
    "learning_rate":     0.05,
    "num_leaves":        127,
    "max_depth":         -1,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "reg_alpha":         0.1,
    "reg_lambda":        0.1,
    "n_jobs":            -1,
    "seed":              SEED,
    "verbose":           -1,
}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=500),
        ],
    )

    oof_pred[val_idx]  = model.predict(X_val)
    oof_lgb[val_idx]  = oof_pred[val_idx]
    test_pred         += model.predict(X_test) / N_SPLITS
    pred_lgb          += model.predict(X_test) / N_SPLITS
    fold_auc           = roc_auc_score(y_val, oof_pred[val_idx])
    fold_aucs.append(fold_auc)
    lgb_models.append(model)

    print(f"  Fold {fold+1}  |  best iter: {model.best_iteration:>4}  |  AUC: {fold_auc:.4f}")

oof_auc = roc_auc_score(y, oof_pred)
print(f"\n  ── 폴드별 AUC: {[round(a, 4) for a in fold_aucs]}")
print(f"  ── AUC 평균 ± 표준편차: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"  ★ OOF AUC (전체): {oof_auc:.4f}")


7. LightGBM 단독 모델 학습 (5-Fold Stratified CV)
  Fold 1  |  best iter:   63  |  AUC: 0.7359
  Fold 2  |  best iter:  105  |  AUC: 0.7416
  Fold 3  |  best iter:   67  |  AUC: 0.7391
  Fold 4  |  best iter:   59  |  AUC: 0.7371
  Fold 5  |  best iter:   75  |  AUC: 0.7386

  ── 폴드별 AUC: [0.7359, 0.7416, 0.7391, 0.7371, 0.7386]
  ── AUC 평균 ± 표준편차: 0.7385 ± 0.0019
  ★ OOF AUC (전체): 0.7384


In [52]:
# ============================================================
# 7. XGBoost 단독 모델 학습 (5-Fold Stratified CV)
# ============================================================
print("\n" + "=" * 60)
print("7. XGBoost 단독 모델 학습 (5-Fold Stratified CV)")
print("=" * 60)

N_SPLITS = 5
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_pred  = np.zeros(len(X_train))
test_pred = np.zeros(len(X_test))
fold_aucs = []
models    = []

xgb_params = {
    "objective":        "binary:logistic",
    "eval_metric":      "auc",
    "learning_rate":    0.05,
    "max_depth":        6,
    "min_child_weight": 5,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "gamma":            0.1,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "seed":             SEED,
    "n_jobs":           -1,
    "tree_method":      "hist",
    "verbosity":        0,
}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]

    dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=list(X_train.columns))
    dval   = xgb.DMatrix(X_val, label=y_val, feature_names=list(X_train.columns))

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, "val")],
        early_stopping_rounds=100,
        verbose_eval=500,
    )

    oof_pred[val_idx]  = model.predict(dval)
    test_pred         += model.predict(
        xgb.DMatrix(X_test, feature_names=list(X_train.columns))
    ) / N_SPLITS
    fold_auc           = roc_auc_score(y_val, oof_pred[val_idx])
    fold_aucs.append(fold_auc)
    models.append(model)

    print(f"  Fold {fold+1}  |  best iter: {model.best_iteration:>4}  |  AUC: {fold_auc:.4f}")

oof_auc = roc_auc_score(y, oof_pred)
print(f"\n  ── 폴드별 AUC: {[round(a, 4) for a in fold_aucs]}")
print(f"  ── AUC 평균 ± 표준편차: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"  ★ OOF AUC (전체): {oof_auc:.4f}")



7. XGBoost 단독 모델 학습 (5-Fold Stratified CV)
[0]	val-auc:0.72528
[240]	val-auc:0.73673
  Fold 1  |  best iter:  140  |  AUC: 0.7367
[0]	val-auc:0.72816
[363]	val-auc:0.74215
  Fold 2  |  best iter:  263  |  AUC: 0.7421
[0]	val-auc:0.72649
[272]	val-auc:0.73970
  Fold 3  |  best iter:  172  |  AUC: 0.7397
[0]	val-auc:0.72647
[251]	val-auc:0.73734
  Fold 4  |  best iter:  151  |  AUC: 0.7373
[0]	val-auc:0.72421
[278]	val-auc:0.73967
  Fold 5  |  best iter:  178  |  AUC: 0.7397

  ── 폴드별 AUC: [0.7367, 0.7421, 0.7397, 0.7373, 0.7397]
  ── AUC 평균 ± 표준편차: 0.7391 ± 0.0019
  ★ OOF AUC (전체): 0.7391


In [53]:
# ============================================================
# 7. CatBoost 단독 모델 학습 (5-Fold Stratified CV)
# ============================================================
print("\n" + "=" * 60)
print("7. CatBoost 단독 모델 학습 (5-Fold Stratified CV)")
print("=" * 60)

N_SPLITS = 5
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_pred  = np.zeros(len(X_train))
test_pred = np.zeros(len(X_test))
fold_aucs = []
cat_models = []

cat_params = {
    "iterations":          2000,
    "learning_rate":       0.05,
    "depth":               6,
    "l2_leaf_reg":         3,
    "eval_metric":         "AUC",
    "early_stopping_rounds": 100,
    "random_seed":         SEED,
    "verbose":             500,
    "task_type":           "CPU",
    "bootstrap_type":      "Bernoulli",
    "subsample":           0.8,
    "colsample_bylevel":   0.8,
}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],       y.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    oof_pred[val_idx]  = model.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = oof_pred[val_idx]
    test_pred         += model.predict_proba(X_test)[:, 1] / N_SPLITS
    pred_cat          += model.predict_proba(X_test)[:, 1] / N_SPLITS
    fold_auc           = roc_auc_score(y_val, oof_pred[val_idx])
    fold_aucs.append(fold_auc)
    cat_models.append(model)

    print(f"  Fold {fold+1}  |  best iter: {model.best_iteration_:>4}  |  AUC: {fold_auc:.4f}")

oof_auc = roc_auc_score(y, oof_pred)
print(f"\n  ── 폴드별 AUC: {[round(a, 4) for a in fold_aucs]}")
print(f"  ── AUC 평균 ± 표준편차: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"  ★ OOF AUC (전체): {oof_auc:.4f}")


7. CatBoost 단독 모델 학습 (5-Fold Stratified CV)
0:	test: 0.7078795	best: 0.7078795 (0)	total: 42.2ms	remaining: 1m 24s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7376803606
bestIteration = 363

Shrink model to first 364 iterations.
  Fold 1  |  best iter:  363  |  AUC: 0.7377
0:	test: 0.7184765	best: 0.7184765 (0)	total: 37.5ms	remaining: 1m 14s
500:	test: 0.7432611	best: 0.7433537 (473)	total: 18.1s	remaining: 54.2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7433537432
bestIteration = 473

Shrink model to first 474 iterations.
  Fold 2  |  best iter:  473  |  AUC: 0.7434
0:	test: 0.7160275	best: 0.7160275 (0)	total: 35.8ms	remaining: 1m 11s
500:	test: 0.7394135	best: 0.7394185 (497)	total: 18.2s	remaining: 54.5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7394535107
bestIteration = 530

Shrink model to first 531 iterations.
  Fold 3  |  best iter:  530  |  AUC: 0.7395
0:	test: 0.7088054	best: 0.7088054 (0)	total

In [54]:
submission = pd.DataFrame({
    "ID": test[ID_COL],
    "probability": test_pred
})

submission.to_csv("submission.csv", index=False)